# 08 · eval + **Ablation 표**

사다리 전체(`acm` → `+carry` → `+BiMamba` → `+MOSAIC`)를 같은 프로토콜로 평가하고
**SR + 경계 떨림**을 한 표에. 각 행이 이전 행 대비 무엇을 더했는지 = 그게 곧 기여.

여기서 새로 도는 건 `07` 에서 학습한 3모델(**60 run**). `acm`(`04`) / `ours`(`02`) 는 이미 평가됨.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM        # 'insertion' (aloha)
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3] — 은지와 분담하면 여기만 바꿈 (예: [0,1])
NGPU  = 8
TAGS   = cf.GROUP_ABLATION          # 여기서 새로 평가할 것
LADDER = cf.ABLATION                # 표에 넣을 사다리 전체 (acm ... ours)
REPS = list(range(cf.EVAL_REPEATS))
N_EP = cf.EVAL_N_EP

print('사다리:', LADDER)
print('여기서 eval:', TAGS, '| run:', len(TAGS) * len(SEEDS) * len(REPS))

## 반복 eval (사다리 나머지 3모델)

In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, ngpu=NGPU, n_episodes=N_EP)

## Ablation 표 — SR + 떨림 → `outputs/final/ablation/ablation.csv`
SR = 150k × 5rep × 4seed (mean±std) / 떨림 = 그 rep 들의 action(.pt) pool.

In [ ]:
import csv
import smooth_metrics as sm

out = cf.OUTPUT_BASE / 'ablation'
out.mkdir(parents=True, exist_ok=True)

fps, K = cf.fps_of(TASK), 100
rows = []
for t in LADDER:
    agg = cf.sr_over_reps(t, task=TASK, seeds=SEEDS, reps=REPS)
    trajs = []
    for s in SEEDS:
        trajs += cf.action_trajs(t, s, TASK, reps=REPS)
    m = sm.aggregate_smoothness(trajs, chunk=K, fps=fps) if trajs else {}
    g = lambda k, d=4: (f'%.{d}f' % m[k]) if k in m else '-'
    rows.append({'tag': t, 'model': cf.v23.MODEL_LABELS.get(t, t),
                 'SR': ('%.1f' % agg['mean']) if agg['mean'] is not None else '-',
                 'SR_std': ('%.1f' % agg['std']) if agg['mean'] is not None else '-',
                 'n_run': agg['n_runs'], 'n_traj': len(trajs),
                 'boundary_jerk': g('boundary_jerk'), 'interior_jerk': g('interior_jerk'),
                 'contrast': g('boundary_jerk_contrast'), 'SPARC': g('sparc', 3)})

with open(out / 'ablation.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

hdr = f"{'MODEL':<34}{'SR':>14}{'b-jerk':>9}{'i-jerk':>9}{'contrast':>10}{'SPARC':>8}"
print(hdr)
print('-' * len(hdr))
for r in rows:
    sr = f"{r['SR']} ± {r['SR_std']}" if r['SR'] != '-' else '-'
    print(f"{r['model']:<34}{sr:>14}{r['boundary_jerk']:>9}{r['interior_jerk']:>9}"
          f"{r['contrast']:>10}{r['SPARC']:>8}")
print('\n저장:', out / 'ablation.csv')